# Amazon ML Challenge 2026 — CatBoost V4 Deployment Build

## Purpose

This notebook is **Kaggle-only**. It does not perform test inference.

It takes the exact training/tuning bundle used by the completed ~0.989 validation experiment and produces a **deployment bundle** for the local inference pipeline.

### Frozen model recipe

- V4 feature set: **18 features**
- Word TF-IDF:
  - word n-grams `(1, 2)`
  - `min_df=2`
  - `max_features=100,000`
- Character TF-IDF:
  - char n-grams `(2, 5)`
  - `min_df=2`
  - `max_features=150,000`
- CatBoost:
  - `iterations=1000`
  - `depth=10`
  - `learning_rate=0.03`
  - `l2_leaf_reg=3.0`
  - `random_seed=42`
  - `loss_function="Logloss"`
- Frozen validation threshold: **0.85**

## Important

The original `.989` notebook saved only the CatBoost estimator. That is **not sufficient for test inference**, because the four TF-IDF vectorizers are part of the feature pipeline.

This notebook therefore packages:

```text
CatBoost model
+ word-name TF-IDF vectorizer
+ word-address TF-IDF vectorizer
+ char-name TF-IDF vectorizer
+ char-address TF-IDF vectorizer
+ exact feature ordering
+ threshold
+ model configuration
```

The resulting deployment artifact is intended to be loaded by the **local inference notebook** against the already-generated candidate partitions.


In [1]:
# ============================================================
# 1. ENVIRONMENT
# ============================================================

from pathlib import Path
import json
import pickle
import re
import time

import joblib
import numpy as np
import pandas as pd
import polars as pl

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, fbeta_score
from catboost import CatBoostClassifier

print("Environment ready.")
print("CatBoost:", __import__("catboost").__version__)


Environment ready.
CatBoost: 1.2.10


In [3]:
# ============================================================
# 2. LOCATE THE EXACT TUNING BUNDLE
# ============================================================

INPUT_DIR = Path("/kaggle/input")

bundle_candidates = list(
    INPUT_DIR.rglob("amazon_ml_tuning_bundle.pkl")
)

print("Candidate bundle paths:")
for p in bundle_candidates:
    print(" ", p)

if not bundle_candidates:
    raise FileNotFoundError(
        "amazon_ml_tuning_bundle.pkl was not found under /kaggle/input."
    )

BUNDLE_PATH = bundle_candidates[0]

print("\nUsing:")
print(BUNDLE_PATH)


Candidate bundle paths:
  /kaggle/input/datasets/abhilashpalit/picklebundle/amazon_ml_tuning_bundle.pkl

Using:
/kaggle/input/datasets/abhilashpalit/picklebundle/amazon_ml_tuning_bundle.pkl


In [4]:
# ============================================================
# 3. LOAD TRAINING / TUNING BUNDLE
# ============================================================

with open(BUNDLE_PATH, "rb") as f:
    bundle = pickle.load(f)

print("Bundle loaded.")
print(f"Bundle size: {BUNDLE_PATH.stat().st_size / 1024**2:.2f} MB")

required_keys = [
    "training_enriched",
    "training_features",
    "train_idx",
    "val_idx",
]

missing = [k for k in required_keys if k not in bundle]

if missing:
    raise RuntimeError(f"Required bundle keys missing: {missing}")

training_enriched = bundle["training_enriched"]
training_features = bundle["training_features"]
train_idx = np.asarray(bundle["train_idx"])
val_idx = np.asarray(bundle["val_idx"])

print("training_enriched:", training_enriched.shape)
print("training_features:", training_features.shape)
print("train_idx:", train_idx.shape)
print("val_idx:", val_idx.shape)


Bundle loaded.
Bundle size: 77.78 MB
training_enriched: (172458, 9)
training_features: (172458, 16)
train_idx: (137966,)
val_idx: (34492,)


In [6]:
# ============================================================
# 4. FIXED-SPLIT INTEGRITY GATE
# ============================================================

if len(np.intersect1d(train_idx, val_idx)) != 0:
    raise RuntimeError("Train/validation overlap detected.")

if len(train_idx) + len(val_idx) != training_enriched.height:
    raise RuntimeError(
        "Train + validation indices do not cover the full training bundle."
    )

if len(np.unique(train_idx)) != len(train_idx):
    raise RuntimeError("Duplicate training indices detected.")

if len(np.unique(val_idx)) != len(val_idx):
    raise RuntimeError("Duplicate validation indices detected.")

print("✓ Fixed split is disjoint.")
print("✓ Fixed split covers all labeled pairs.")
print(f"Train rows: {len(train_idx):,}")
print(f"Validation rows: {len(val_idx):,}")
print(f"Total rows: {training_enriched.height:,}")


✓ Fixed split is disjoint.
✓ Fixed split covers all labeled pairs.
Train rows: 137,966
Validation rows: 34,492
Total rows: 172,458


In [8]:
# ============================================================
# 5. EXACT V2 FEATURE FUNCTIONS
# ============================================================

def normalize_text(value):
    if value is None:
        return ""

    value = str(value).lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def normalize_legal_name(value):
    value = normalize_text(value)

    legal_suffixes = [
        "private limited",
        "pvt ltd",
        "pvt limited",
        "private ltd",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corp",
        "corporation",
        "company",
        "co",
    ]

    tokens = value.split()

    while tokens:
        removed = False

        for suffix in legal_suffixes:
            suffix_tokens = suffix.split()

            if (
                len(tokens) >= len(suffix_tokens)
                and tokens[-len(suffix_tokens):] == suffix_tokens
            ):
                tokens = tokens[:-len(suffix_tokens)]
                removed = True
                break

        if not removed:
            break

    return " ".join(tokens)


def safe_length_ratio(a, b):
    a = a or ""
    b = b or ""

    if not a or not b:
        return 0.0

    return min(len(a), len(b)) / max(len(a), len(b))


def exact_normalized(a, b):
    a = normalize_text(a)
    b = normalize_text(b)

    if not a or not b:
        return 0

    return int(a == b)


def exact_legal_name(a, b):
    a = normalize_legal_name(a)
    b = normalize_legal_name(b)

    if not a or not b:
        return 0

    return int(a == b)


V1_FEATURES = [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match",
]

V2_NEW_FEATURES = [
    "name_exact_normalized",
    "name_exact_legal",
    "name_length_ratio",
    "address_exact_normalized",
    "address_length_ratio",
    "s1_name_missing",
    "match_name_missing",
    "s1_address_missing",
    "match_address_missing",
]

V2_FEATURES = V1_FEATURES + V2_NEW_FEATURES

print("V2 feature functions ready.")
print("V2 feature count:", len(V2_FEATURES))


V2 feature functions ready.
V2 feature count: 14


In [9]:
# ============================================================
# 6. REBUILD V2 FEATURES EXACTLY
# ============================================================

v2 = training_enriched.with_columns([

    pl.struct(["s1_name", "match_name"])
    .map_elements(
        lambda x: exact_normalized(x["s1_name"], x["match_name"]),
        return_dtype=pl.Int8,
    )
    .alias("name_exact_normalized"),

    pl.struct(["s1_name", "match_name"])
    .map_elements(
        lambda x: exact_legal_name(x["s1_name"], x["match_name"]),
        return_dtype=pl.Int8,
    )
    .alias("name_exact_legal"),

    pl.struct(["s1_name", "match_name"])
    .map_elements(
        lambda x: safe_length_ratio(x["s1_name"], x["match_name"]),
        return_dtype=pl.Float64,
    )
    .alias("name_length_ratio"),

    pl.struct(["s1_address", "match_address"])
    .map_elements(
        lambda x: exact_normalized(x["s1_address"], x["match_address"]),
        return_dtype=pl.Int8,
    )
    .alias("address_exact_normalized"),

    pl.struct(["s1_address", "match_address"])
    .map_elements(
        lambda x: safe_length_ratio(x["s1_address"], x["match_address"]),
        return_dtype=pl.Float64,
    )
    .alias("address_length_ratio"),

    (
        pl.col("s1_name").fill_null("").str.strip_chars() == ""
    ).cast(pl.Int8).alias("s1_name_missing"),

    (
        pl.col("match_name").fill_null("").str.strip_chars() == ""
    ).cast(pl.Int8).alias("match_name_missing"),

    (
        pl.col("s1_address").fill_null("").str.strip_chars() == ""
    ).cast(pl.Int8).alias("s1_address_missing"),

    (
        pl.col("match_address").fill_null("").str.strip_chars() == ""
    ).cast(pl.Int8).alias("match_address_missing"),
])

v2_feature_table = v2.with_columns([
    training_features[col]
    for col in V1_FEATURES
])

missing = [c for c in V2_FEATURES if c not in v2_feature_table.columns]

if missing:
    raise RuntimeError(f"Missing V2 features: {missing}")

print("V2 rows:", v2_feature_table.height)
print("V2 features:", len(V2_FEATURES))
print("✓ V2 feature reconstruction passed.")


V2 rows: 172458
V2 features: 14
✓ V2 feature reconstruction passed.


In [10]:
# ============================================================
# 7. PREPARE TEXT ARRAYS
# ============================================================

train_idx_np = train_idx
val_idx_np = val_idx

s1_names = (
    training_enriched["s1_name"]
    .fill_null("")
    .to_numpy()
)

match_names = (
    training_enriched["match_name"]
    .fill_null("")
    .to_numpy()
)

s1_addresses = (
    training_enriched["s1_address"]
    .fill_null("")
    .to_numpy()
)

match_addresses = (
    training_enriched["match_address"]
    .fill_null("")
    .to_numpy()
)

train_s1_names = s1_names[train_idx_np]
train_match_names = match_names[train_idx_np]
val_s1_names = s1_names[val_idx_np]
val_match_names = match_names[val_idx_np]

train_s1_addresses = s1_addresses[train_idx_np]
train_match_addresses = match_addresses[train_idx_np]
val_s1_addresses = s1_addresses[val_idx_np]
val_match_addresses = match_addresses[val_idx_np]

print("Train text rows:", len(train_s1_names))
print("Validation text rows:", len(val_s1_names))


Train text rows: 137966
Validation text rows: 34492


In [11]:
# ============================================================
# 8. VALIDATION REPRODUCTION: FIT WORD TF-IDF ON TRAIN SPLIT
# ============================================================

name_vectorizer_val = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)

address_vectorizer_val = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)

name_vectorizer_val.fit(
    list(train_s1_names) + list(train_match_names)
)

address_vectorizer_val.fit(
    list(train_s1_addresses) + list(train_match_addresses)
)

print("Name vocabulary:", len(name_vectorizer_val.vocabulary_))
print("Address vocabulary:", len(address_vectorizer_val.vocabulary_))


Name vocabulary: 42731
Address vocabulary: 100000


In [12]:
# ============================================================
# 9. VALIDATION REPRODUCTION: FIT CHARACTER TF-IDF
# ============================================================

char_name_vectorizer_val = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000,
)

char_address_vectorizer_val = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000,
)

char_name_vectorizer_val.fit(
    list(train_s1_names) + list(train_match_names)
)

char_address_vectorizer_val.fit(
    list(train_s1_addresses) + list(train_match_addresses)
)

print("Name char vocabulary:", len(char_name_vectorizer_val.vocabulary_))
print("Address char vocabulary:", len(char_address_vectorizer_val.vocabulary_))


Name char vocabulary: 150000
Address char vocabulary: 150000


In [13]:
# ============================================================
# 10. VALIDATION REPRODUCTION: TF-IDF COSINE FEATURES
# ============================================================

def rowwise_cosine_similarity(A, B):
    # TfidfVectorizer normalizes rows by default, so dot product
    # is cosine similarity for corresponding rows.
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()


train_name_s1_tfidf = name_vectorizer_val.transform(train_s1_names)
train_name_match_tfidf = name_vectorizer_val.transform(train_match_names)
val_name_s1_tfidf = name_vectorizer_val.transform(val_s1_names)
val_name_match_tfidf = name_vectorizer_val.transform(val_match_names)

train_address_s1_tfidf = address_vectorizer_val.transform(train_s1_addresses)
train_address_match_tfidf = address_vectorizer_val.transform(train_match_addresses)
val_address_s1_tfidf = address_vectorizer_val.transform(val_s1_addresses)
val_address_match_tfidf = address_vectorizer_val.transform(val_match_addresses)

train_char_name_s1 = char_name_vectorizer_val.transform(train_s1_names)
train_char_name_match = char_name_vectorizer_val.transform(train_match_names)
val_char_name_s1 = char_name_vectorizer_val.transform(val_s1_names)
val_char_name_match = char_name_vectorizer_val.transform(val_match_names)

train_char_address_s1 = char_address_vectorizer_val.transform(train_s1_addresses)
train_char_address_match = char_address_vectorizer_val.transform(train_match_addresses)
val_char_address_s1 = char_address_vectorizer_val.transform(val_s1_addresses)
val_char_address_match = char_address_vectorizer_val.transform(val_match_addresses)

train_name_tfidf_cosine = rowwise_cosine_similarity(
    train_name_s1_tfidf, train_name_match_tfidf
)
val_name_tfidf_cosine = rowwise_cosine_similarity(
    val_name_s1_tfidf, val_name_match_tfidf
)

train_address_tfidf_cosine = rowwise_cosine_similarity(
    train_address_s1_tfidf, train_address_match_tfidf
)
val_address_tfidf_cosine = rowwise_cosine_similarity(
    val_address_s1_tfidf, val_address_match_tfidf
)

train_name_char_tfidf_cosine = rowwise_cosine_similarity(
    train_char_name_s1, train_char_name_match
)
val_name_char_tfidf_cosine = rowwise_cosine_similarity(
    val_char_name_s1, val_char_name_match
)

train_address_char_tfidf_cosine = rowwise_cosine_similarity(
    train_char_address_s1, train_char_address_match
)
val_address_char_tfidf_cosine = rowwise_cosine_similarity(
    val_char_address_s1, val_char_address_match
)

print("✓ Validation TF-IDF similarities calculated.")


✓ Validation TF-IDF similarities calculated.


In [15]:
# ============================================================
# 11. BUILD EXACT V4 VALIDATION MATRICES
# ============================================================

X_v2 = (
    v2_feature_table
    .select(V2_FEATURES)
    .to_pandas()
)

y_v2 = v2_feature_table["label"].to_pandas()

X_train_v2 = X_v2.iloc[train_idx].copy()
X_val_v2 = X_v2.iloc[val_idx].copy()

y_train_v2 = y_v2.iloc[train_idx].copy()
y_val_v2 = y_v2.iloc[val_idx].copy()

X_train_v4 = np.column_stack([
    X_train_v2.to_numpy(),
    train_name_tfidf_cosine,
    train_address_tfidf_cosine,
    train_name_char_tfidf_cosine,
    train_address_char_tfidf_cosine,
])

X_val_v4 = np.column_stack([
    X_val_v2.to_numpy(),
    val_name_tfidf_cosine,
    val_address_tfidf_cosine,
    val_name_char_tfidf_cosine,
    val_address_char_tfidf_cosine,
])

V4_FEATURES = V2_FEATURES + [
    "name_tfidf_cosine",
    "address_tfidf_cosine",
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine",
]

if len(V4_FEATURES) != 18:
    raise RuntimeError("V4 feature count is not 18.")

if X_train_v4.shape[1] != 18 or X_val_v4.shape[1] != 18:
    raise RuntimeError("V4 matrix does not have 18 columns.")

if not np.isfinite(X_train_v4).all():
    raise RuntimeError("Non-finite values found in X_train_v4.")

if not np.isfinite(X_val_v4).all():
    raise RuntimeError("Non-finite values found in X_val_v4.")

print("X_train_v4:", X_train_v4.shape)
print("X_val_v4  :", X_val_v4.shape)
print("✓ Exact 18-feature V4 matrices built.")


X_train_v4: (137966, 18)
X_val_v4  : (34492, 18)
✓ Exact 18-feature V4 matrices built.


In [16]:
# ============================================================
# 12. REPRODUCE THE .989 CATBOOST RESULT
# ============================================================

VALIDATION_THRESHOLD = 0.85

cat_v4_validation = CatBoostClassifier(
    iterations=1000,
    depth=10,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Logloss",
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
    thread_count=-1,
)

print("Training validation reproduction model...")
cat_v4_validation.fit(X_train_v4, y_train_v2)

val_prob = cat_v4_validation.predict_proba(X_val_v4)[:, 1]
val_pred = (val_prob >= VALIDATION_THRESHOLD).astype(int)

val_precision = precision_score(
    y_val_v2, val_pred, zero_division=0
)
val_recall = recall_score(
    y_val_v2, val_pred, zero_division=0
)
val_f05 = fbeta_score(
    y_val_v2, val_pred, beta=0.5, zero_division=0
)

print("=" * 65)
print("V4 CATBOOST DEPTH-10 VALIDATION REPRODUCTION")
print("=" * 65)
print(f"Threshold : {VALIDATION_THRESHOLD:.2f}")
print(f"Precision : {val_precision:.6%}")
print(f"Recall    : {val_recall:.6%}")
print(f"F0.5      : {val_f05:.6%}")
print(f"Predicted : {int(val_pred.sum()):,}")
print("=" * 65)

# Guard against accidentally deploying a materially different recipe.
if val_f05 < 0.985:
    raise RuntimeError(
        f"Validation reproduction is unexpectedly low: F0.5={val_f05:.6f}. "
        "STOP before production training and inspect the feature pipeline."
    )

print("✓ Validation reproduction passed the safety gate.")


Training validation reproduction model...
V4 CATBOOST DEPTH-10 VALIDATION REPRODUCTION
Threshold : 0.85
Precision : 99.255084%
Recall    : 97.565802%
F0.5      : 98.912564%
Predicted : 4,967
✓ Validation reproduction passed the safety gate.


============================================================
GROUPKFOLD OOF AUDIT
============================================================

In [17]:
# ============================================================
# GROUPKFOLD OOF AUDIT
# Exact V4 + CatBoost Depth-10 recipe
# ============================================================

import numpy as np
import polars as pl

from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, fbeta_score
from catboost import CatBoostClassifier

print("=" * 70)
print("GROUPKFOLD DIAGNOSTIC SETUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. V2 base features
# ------------------------------------------------------------

V2_FEATURES = [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match",

    "name_exact_normalized",
    "name_exact_legal",
    "name_length_ratio",

    "address_exact_normalized",
    "address_length_ratio",

    "s1_name_missing",
    "match_name_missing",
    "s1_address_missing",
    "match_address_missing",
]

# ------------------------------------------------------------
# 2. V4 TF-IDF feature names
# ------------------------------------------------------------

TFIDF_FEATURES = [
    "name_word_tfidf_cosine",
    "address_word_tfidf_cosine",
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine",
]

V4_FEATURES = V2_FEATURES + TFIDF_FEATURES

print("V2 features :", len(V2_FEATURES))
print("TF-IDF feats :", len(TFIDF_FEATURES))
print("V4 features :", len(V4_FEATURES))

assert len(V4_FEATURES) == 18

# ------------------------------------------------------------
# 3. Verify source table
# ------------------------------------------------------------

required_columns = [
    "source1_entity_id",
    "label",
    "s1_name",
    "s1_address",
    "match_name",
    "match_address",
]

missing = [
    c for c in required_columns
    if c not in training_enriched.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("\nTraining enriched shape:")
print(training_enriched.shape)

print("\nUnique S1 groups:")
print(training_enriched["source1_entity_id"].n_unique())

print("\nPositive pairs:")
print(training_enriched["label"].sum())

print("\nNegative pairs:")
print(
    training_enriched.height
    - training_enriched["label"].sum()
)

print("\n✓ Diagnostic setup passed.")

GROUPKFOLD DIAGNOSTIC SETUP
V2 features : 14
TF-IDF feats : 4
V4 features : 18

Training enriched shape:
(172458, 9)

Unique S1 groups:
5535

Positive pairs:
25267

Negative pairs:
147191

✓ Diagnostic setup passed.


In [18]:
# ============================================================
# TF-IDF COSINE HELPER
# ============================================================

def tfidf_cosine_features(
    train_s1,
    train_match,
    val_s1,
    val_match,
    analyzer,
    ngram_range,
    min_df,
    max_features,
):
    """
    Fit TF-IDF ONLY on the fold's training text,
    then transform both train and validation text.

    Returns:
        train_cosine
        val_cosine
        vectorizer
    """

    # Combine the two sides of the training pairs
    fit_text = (
        list(train_s1)
        + list(train_match)
    )

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
    )

    vectorizer.fit(fit_text)

    # Transform train
    X_train_s1 = vectorizer.transform(train_s1)
    X_train_match = vectorizer.transform(train_match)

    # Transform validation
    X_val_s1 = vectorizer.transform(val_s1)
    X_val_match = vectorizer.transform(val_match)

    # Element-wise dot product because TF-IDF vectors
    # are L2 normalized by default.
    train_cosine = np.asarray(
        X_train_s1.multiply(X_train_match).sum(axis=1)
    ).ravel()

    val_cosine = np.asarray(
        X_val_s1.multiply(X_val_match).sum(axis=1)
    ).ravel()

    return (
        train_cosine.astype(np.float32),
        val_cosine.astype(np.float32),
        vectorizer,
    )


print("✓ TF-IDF helper defined.")

✓ TF-IDF helper defined.


In [19]:
# ============================================================
# REBUILD MISSING V2 FEATURES
# ============================================================

import numpy as np
import polars as pl
import re


def normalize_text(s):
    """
    Basic normalized representation used for exact-match features.
    """
    if s is None:
        return ""
    
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    
    return s.strip()


def normalize_legal_name(s):
    """
    Normalize business name while removing common legal suffixes.
    """
    s = normalize_text(s)
    
    legal_suffixes = [
        "private limited",
        "pvt ltd",
        "pvt limited",
        "limited",
        "ltd",
        "llc",
        "incorporated",
        "inc",
        "corp",
        "corporation",
        "co",
        "company"
    ]
    
    for suffix in legal_suffixes:
        pattern = rf"\b{re.escape(suffix)}\b"
        s = re.sub(pattern, " ", s)
    
    s = re.sub(r"\s+", " ", s)
    
    return s.strip()


def length_ratio(a, b):
    """
    Symmetric length ratio:
        min(len(a), len(b)) / max(len(a), len(b))
    """
    a = "" if a is None else str(a)
    b = "" if b is None else str(b)
    
    la = len(a)
    lb = len(b)
    
    if la == 0 and lb == 0:
        return 1.0
    
    if la == 0 or lb == 0:
        return 0.0
    
    return min(la, lb) / max(la, lb)


# ------------------------------------------------------------
# Build V2 features from the existing raw columns
# ------------------------------------------------------------

training_features = (
    training_features
    .with_columns([
        
        # -------------------------
        # Normalized names
        # -------------------------
        pl.col("s1_name")
        .map_elements(normalize_text, return_dtype=pl.String)
        .alias("_s1_name_norm"),
        
        pl.col("match_name")
        .map_elements(normalize_text, return_dtype=pl.String)
        .alias("_match_name_norm"),
        
        # -------------------------
        # Legal-normalized names
        # -------------------------
        pl.col("s1_name")
        .map_elements(normalize_legal_name, return_dtype=pl.String)
        .alias("_s1_name_legal"),
        
        pl.col("match_name")
        .map_elements(normalize_legal_name, return_dtype=pl.String)
        .alias("_match_name_legal"),
        
        # -------------------------
        # Missingness
        # -------------------------
        pl.col("s1_name").is_null().alias("s1_name_missing"),
        pl.col("match_name").is_null().alias("match_name_missing"),
        pl.col("s1_address").is_null().alias("s1_address_missing"),
        pl.col("match_address").is_null().alias("match_address_missing"),
    ])
    
    .with_columns([
        
        # -------------------------
        # Exact normalized name
        # -------------------------
        (
            (pl.col("_s1_name_norm") == pl.col("_match_name_norm"))
            &
            (pl.col("_s1_name_norm") != "")
        )
        .cast(pl.Int8)
        .alias("name_exact_normalized"),
        
        # -------------------------
        # Exact legal-normalized name
        # -------------------------
        (
            (pl.col("_s1_name_legal") == pl.col("_match_name_legal"))
            &
            (pl.col("_s1_name_legal") != "")
        )
        .cast(pl.Int8)
        .alias("name_exact_legal"),
        
        # -------------------------
        # Name length ratio
        # -------------------------
        pl.struct([
            pl.col("s1_name"),
            pl.col("match_name")
        ])
        .map_elements(
            lambda x: length_ratio(
                x["s1_name"],
                x["match_name"]
            ),
            return_dtype=pl.Float32
        )
        .alias("name_length_ratio"),
        
        # -------------------------
        # Normalized addresses
        # -------------------------
        pl.struct([
            pl.col("s1_address"),
            pl.col("match_address")
        ])
        .map_elements(
            lambda x: (
                normalize_text(x["s1_address"])
                == normalize_text(x["match_address"])
                and
                normalize_text(x["s1_address"]) != ""
            ),
            return_dtype=pl.Boolean
        )
        .cast(pl.Int8)
        .alias("address_exact_normalized"),
        
        # -------------------------
        # Address length ratio
        # -------------------------
        pl.struct([
            pl.col("s1_address"),
            pl.col("match_address")
        ])
        .map_elements(
            lambda x: length_ratio(
                x["s1_address"],
                x["match_address"]
            ),
            return_dtype=pl.Float32
        )
        .alias("address_length_ratio"),
    ])
)


print("V2 feature reconstruction complete.")
print("Shape:", training_features.shape)

print("\nV2 feature availability:")
for feature in V2_FEATURES:
    print(
        f"{feature:<30}",
        feature in training_features.columns
    )

V2 feature reconstruction complete.
Shape: (172458, 29)

V2 feature availability:
name_edit_similarity           True
name_token_jaccard             True
address_edit_similarity        True
address_token_jaccard          True
country_match                  True
name_exact_normalized          True
name_exact_legal               True
name_length_ratio              True
address_exact_normalized       True
address_length_ratio           True
s1_name_missing                True
match_name_missing             True
s1_address_missing             True
match_address_missing          True


In [20]:
# ============================================================
# V2 FEATURE SAFETY CHECK
# ============================================================

missing_v2 = [
    f for f in V2_FEATURES
    if f not in training_features.columns
]

print("V2 FEATURES:")
print(V2_FEATURES)

print("\nMissing:")
print(missing_v2)

assert not missing_v2, (
    f"V2 feature reconstruction failed. "
    f"Missing: {missing_v2}"
)

print("\n✓ All 14 V2 features are present.")

V2 FEATURES:
['name_edit_similarity', 'name_token_jaccard', 'address_edit_similarity', 'address_token_jaccard', 'country_match', 'name_exact_normalized', 'name_exact_legal', 'name_length_ratio', 'address_exact_normalized', 'address_length_ratio', 's1_name_missing', 'match_name_missing', 's1_address_missing', 'match_address_missing']

Missing:
[]

✓ All 14 V2 features are present.


In [21]:
# ============================================================
# GROUPKFOLD OOF
# EXACT V4 + CATBOOST DEPTH-10 RECIPE
# ============================================================

groups = training_enriched["source1_entity_id"].to_numpy()
y = training_enriched["label"].to_numpy().astype(np.int8)

# Base V2 numerical features
base_features = (
    training_features
    .select(V2_FEATURES)
    .to_numpy()
    .astype(np.float32)
)

assert base_features.shape == (
    training_enriched.height,
    14,
)

print("=" * 70)
print("GROUPKFOLD OOF TRAINING")
print("=" * 70)

print("Base feature matrix:", base_features.shape)
print("Groups:", len(np.unique(groups)))
print("Rows:", len(y))

N_SPLITS = 5

gkf = GroupKFold(
    n_splits=N_SPLITS
)

oof_probability = np.full(
    len(y),
    np.nan,
    dtype=np.float32
)

oof_fold = np.full(
    len(y),
    -1,
    dtype=np.int8
)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        base_features,
        y,
        groups=groups
    ),
    start=1
):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    print("Train rows:", len(train_idx))
    print("Val rows  :", len(val_idx))

    # --------------------------------------------------------
    # Safety check: no S1 group overlap
    # --------------------------------------------------------

    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])

    overlap = train_groups.intersection(val_groups)

    print("S1 group overlap:", len(overlap))

    assert len(overlap) == 0

    # --------------------------------------------------------
    # Base V2 features
    # --------------------------------------------------------

    X_train_v2 = base_features[train_idx]
    X_val_v2 = base_features[val_idx]

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    fold_train = training_enriched[train_idx]
    fold_val = training_enriched[val_idx]

    train_s1_name = (
        fold_train["s1_name"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    train_match_name = (
        fold_train["match_name"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    val_s1_name = (
        fold_val["s1_name"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    val_match_name = (
        fold_val["match_name"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    train_s1_address = (
        fold_train["s1_address"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    train_match_address = (
        fold_train["match_address"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    val_s1_address = (
        fold_val["s1_address"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    val_match_address = (
        fold_val["match_address"]
        .fill_null("")
        .cast(pl.String)
        .to_list()
    )

    # --------------------------------------------------------
    # WORD TF-IDF
    # Exact V4 recipe
    # --------------------------------------------------------

    (
        train_name_word,
        val_name_word,
        _
    ) = tfidf_cosine_features(
        train_s1_name,
        train_match_name,
        val_s1_name,
        val_match_name,
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=100_000,
    )

    (
        train_address_word,
        val_address_word,
        _
    ) = tfidf_cosine_features(
        train_s1_address,
        train_match_address,
        val_s1_address,
        val_match_address,
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=100_000,
    )

    # --------------------------------------------------------
    # CHARACTER TF-IDF
    # Exact V4 recipe
    # --------------------------------------------------------

    (
        train_name_char,
        val_name_char,
        _
    ) = tfidf_cosine_features(
        train_s1_name,
        train_match_name,
        val_s1_name,
        val_match_name,
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=150_000,
    )

    (
        train_address_char,
        val_address_char,
        _
    ) = tfidf_cosine_features(
        train_s1_address,
        train_match_address,
        val_s1_address,
        val_match_address,
        analyzer="char",
        ngram_range=(2, 5),
        min_df=2,
        max_features=150_000,
    )

    # --------------------------------------------------------
    # Assemble exact 18-feature V4 matrices
    # --------------------------------------------------------

    X_train_v4 = np.column_stack([
        X_train_v2,
        train_name_word,
        train_address_word,
        train_name_char,
        train_address_char,
    ]).astype(np.float32)

    X_val_v4 = np.column_stack([
        X_val_v2,
        val_name_word,
        val_address_word,
        val_name_char,
        val_address_char,
    ]).astype(np.float32)

    print("X_train_v4:", X_train_v4.shape)
    print("X_val_v4  :", X_val_v4.shape)

    assert X_train_v4.shape[1] == 18
    assert X_val_v4.shape[1] == 18

    # --------------------------------------------------------
    # Exact CatBoost V4 configuration
    # --------------------------------------------------------

    model = CatBoostClassifier(
        iterations=1000,
        depth=10,
        learning_rate=0.03,
        l2_leaf_reg=3,
        random_seed=42,
        loss_function="Logloss",
        verbose=False,
        allow_writing_files=False,
    )

    print("Training CatBoost...")

    model.fit(
        X_train_v4,
        y[train_idx],
    )

    # --------------------------------------------------------
    # OOF probabilities
    # --------------------------------------------------------

    val_probability = model.predict_proba(
        X_val_v4
    )[:, 1]

    oof_probability[val_idx] = val_probability
    oof_fold[val_idx] = fold

    # --------------------------------------------------------
    # Fold diagnostic at original threshold 0.85
    # --------------------------------------------------------

    fold_pred = (
        val_probability >= 0.85
    ).astype(np.int8)

    fold_precision = precision_score(
        y[val_idx],
        fold_pred,
        zero_division=0,
    )

    fold_recall = recall_score(
        y[val_idx],
        fold_pred,
        zero_division=0,
    )

    fold_f05 = fbeta_score(
        y[val_idx],
        fold_pred,
        beta=0.5,
        zero_division=0,
    )

    fold_results.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "precision@0.85": fold_precision,
        "recall@0.85": fold_recall,
        "f0.5@0.85": fold_f05,
    })

    print(
        f"Fold {fold} @ 0.85:"
    )
    print(
        f"  Precision = {fold_precision:.6f}"
    )
    print(
        f"  Recall    = {fold_recall:.6f}"
    )
    print(
        f"  F0.5      = {fold_f05:.6f}"
    )

print("\n" + "=" * 70)
print("OOF TRAINING COMPLETE")
print("=" * 70)

assert not np.isnan(oof_probability).any()
assert not (oof_fold == -1).any()

print("✓ Every row has exactly one OOF prediction.")
print("OOF rows:", len(oof_probability))

GROUPKFOLD OOF TRAINING
Base feature matrix: (172458, 14)
Groups: 5535
Rows: 172458

FOLD 1/5
Train rows: 137966
Val rows  : 34492
S1 group overlap: 0
X_train_v4: (137966, 18)
X_val_v4  : (34492, 18)
Training CatBoost...
Fold 1 @ 0.85:
  Precision = 0.992933
  Recall    = 0.981374
  F0.5      = 0.990599

FOLD 2/5
Train rows: 137966
Val rows  : 34492
S1 group overlap: 0
X_train_v4: (137966, 18)
X_val_v4  : (34492, 18)
Training CatBoost...
Fold 2 @ 0.85:
  Precision = 0.993490
  Recall    = 0.970141
  F0.5      = 0.988731

FOLD 3/5
Train rows: 137966
Val rows  : 34492
S1 group overlap: 0
X_train_v4: (137966, 18)
X_val_v4  : (34492, 18)
Training CatBoost...
Fold 3 @ 0.85:
  Precision = 0.995740
  Recall    = 0.978278
  F0.5      = 0.992198

FOLD 4/5
Train rows: 137967
Val rows  : 34491
S1 group overlap: 0
X_train_v4: (137967, 18)
X_val_v4  : (34491, 18)
Training CatBoost...
Fold 4 @ 0.85:
  Precision = 0.992340
  Recall    = 0.982373
  F0.5      = 0.990330

FOLD 5/5
Train rows: 137967
Val

In [27]:
# ============================================================
# ENTITY-LEVEL MACRO F0.5
# FIXED VERSION
# ============================================================

def entity_macro_f05(df, threshold):

    # --------------------------------------------------------
    # 1. Predicted matches at this threshold
    # --------------------------------------------------------
    pred = (
        df
        .filter(pl.col("probability") >= threshold)
        .select([
            "source1_entity_id",
            "match_id"
        ])
    )

    # --------------------------------------------------------
    # 2. Ground-truth matches
    # --------------------------------------------------------
    true_by_s1 = (
        df
        .filter(pl.col("label") == 1)
        .group_by("source1_entity_id")
        .agg(
            pl.col("match_id")
            .unique()
            .alias("true_ids")
        )
    )

    # --------------------------------------------------------
    # 3. Predicted matches
    # --------------------------------------------------------
    pred_by_s1 = (
        pred
        .group_by("source1_entity_id")
        .agg(
            pl.col("match_id")
            .unique()
            .alias("pred_ids")
        )
    )

    # --------------------------------------------------------
    # 4. Start from ALL S1 entities
    # --------------------------------------------------------
    all_s1 = (
        df
        .select("source1_entity_id")
        .unique()
    )

    joined = (
        all_s1
        .join(
            true_by_s1,
            on="source1_entity_id",
            how="left"
        )
        .join(
            pred_by_s1,
            on="source1_entity_id",
            how="left"
        )
    )

    # --------------------------------------------------------
    # 5. Calculate entity-level F0.5
    #
    # IMPORTANT:
    # None means that S1 has no true/predicted IDs.
    # We handle that in Python instead of forcing Polars
    # to construct an empty String list.
    # --------------------------------------------------------

    scores = []

    for row in joined.iter_rows(named=True):

        true_ids = set(
            row["true_ids"]
            if row["true_ids"] is not None
            else []
        )

        pred_ids = set(
            row["pred_ids"]
            if row["pred_ids"] is not None
            else []
        )

        tp = len(true_ids & pred_ids)
        fp = len(pred_ids - true_ids)
        fn = len(true_ids - pred_ids)

        # ----------------------------------------------------
        # Entity has no true matches and model predicts none.
        # Correct singleton.
        # ----------------------------------------------------
        if tp == 0 and fp == 0 and fn == 0:

            f05 = 1.0

        # ----------------------------------------------------
        # Model predicted something incorrectly.
        # ----------------------------------------------------
        elif tp == 0:

            f05 = 0.0

        # ----------------------------------------------------
        # Normal F0.5 calculation
        # ----------------------------------------------------
        else:

            precision = tp / (tp + fp)
            recall = tp / (tp + fn)

            beta = 0.5

            f05 = (
                (1 + beta**2)
                * precision
                * recall
                / (
                    beta**2 * precision
                    + recall
                )
            )

        scores.append(f05)

    # --------------------------------------------------------
    # 6. Macro average across S1 entities
    # --------------------------------------------------------

    return float(np.mean(scores))

In [28]:
# ============================================================
# ENTITY-LEVEL THRESHOLD SWEEP
# ============================================================

thresholds = np.arange(
    0.50,
    0.991,
    0.01
)

results = []

for threshold in thresholds:

    score = entity_macro_f05(
        oof_eval,
        threshold
    )

    results.append({
        "threshold": threshold,
        "macro_f05": score
    })

threshold_results = (
    pl.DataFrame(results)
    .sort("macro_f05", descending=True)
)

print(threshold_results.head(15))

shape: (15, 2)
┌───────────┬───────────┐
│ threshold ┆ macro_f05 │
│ ---       ┆ ---       │
│ f64       ┆ f64       │
╞═══════════╪═══════════╡
│ 0.8       ┆ 0.98444   │
│ 0.78      ┆ 0.984419  │
│ 0.83      ┆ 0.984416  │
│ 0.84      ┆ 0.984413  │
│ 0.81      ┆ 0.98435   │
│ …         ┆ …         │
│ 0.76      ┆ 0.984217  │
│ 0.87      ┆ 0.984124  │
│ 0.85      ┆ 0.984111  │
│ 0.86      ┆ 0.983905  │
│ 0.9       ┆ 0.983803  │
└───────────┴───────────┘


In [29]:
best_row = threshold_results.row(0, named=True)

print("BEST ENTITY-LEVEL THRESHOLD")
print("=" * 50)
print("Threshold :", best_row["threshold"])
print("Macro F0.5 :", best_row["macro_f05"])

BEST ENTITY-LEVEL THRESHOLD
Threshold : 0.8000000000000003
Macro F0.5 : 0.984440136738672


In [30]:
# ============================================================
# LEADERBOARD-PARITY VALIDATION
# STEP 1: S1-LEVEL HOLDOUT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import polars as pl

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20

groups = training_enriched["source1_entity_id"].to_numpy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE
)

train_idx_g, holdout_idx_g = next(
    gss.split(
        np.zeros(len(groups)),
        training_enriched["label"].to_numpy(),
        groups=groups
    )
)

train_s1 = set(
    training_enriched
    .select("source1_entity_id")
    .to_series()
    .to_list()
)

holdout_s1 = set(
    training_enriched
    .select("source1_entity_id")
    .to_series()
    .to_list()
)

# Correctly derive the actual group split
train_s1 = set(
    groups[train_idx_g]
)

holdout_s1 = set(
    groups[holdout_idx_g]
)

print("=" * 65)
print("S1-LEVEL HOLDOUT")
print("=" * 65)

print("Total S1 groups :", len(set(groups)))
print("Train S1 groups :", len(train_s1))
print("Holdout S1      :", len(holdout_s1))

print()
print("Pair rows in train   :", len(train_idx_g))
print("Pair rows in holdout :", len(holdout_idx_g))

assert train_s1.isdisjoint(holdout_s1)

print()
print("✓ ZERO S1 GROUP OVERLAP")

S1-LEVEL HOLDOUT
Total S1 groups : 5535
Train S1 groups : 4428
Holdout S1      : 1107

Pair rows in train   : 137743
Pair rows in holdout : 34715

✓ ZERO S1 GROUP OVERLAP


In [34]:
# ============================================================
# STEP 2: IDENTIFY FULL DEVELOPMENT CANDIDATE TABLE
# ============================================================

print("Candidate objects currently available:")
print()

for name in [
    "final_candidates",
    "union_candidates",
    "candidate_pairs",
    "all_candidates",
    "training_candidates"
]:
    if name in globals():
        obj = globals()[name]

        print(
            f"{name:25s}",
            type(obj),
            end=""
        )

        try:
            print(
                " | shape:",
                obj.shape
            )
        except Exception:
            print()

Candidate objects currently available:



In [35]:
# ============================================================
# STEP 2B: FIND AVAILABLE CANDIDATE ARTIFACTS
# ============================================================

from pathlib import Path

print("=" * 65)
print("SEARCHING FOR CANDIDATE ARTIFACTS")
print("=" * 65)

# Search common working locations
search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

candidate_keywords = [
    "candidate",
    "candidates",
    "union",
    "pair",
    "pairs",
]

found = []

for root in search_roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        name = p.name.lower()

        if any(k in name for k in candidate_keywords):
            try:
                size_mb = p.stat().st_size / (1024 ** 2)
            except:
                size_mb = -1

            found.append((str(p), size_mb))

            print(
                f"  {p.name:45s} "
                f"{size_mb:10.2f} MB"
            )

print("\n" + "=" * 65)
print(f"TOTAL CANDIDATE-LIKE FILES FOUND: {len(found)}")
print("=" * 65)

SEARCHING FOR CANDIDATE ARTIFACTS

Searching: /kaggle/working

Searching: /kaggle/input

TOTAL CANDIDATE-LIKE FILES FOUND: 0


## Production build

The validation experiment above is deliberately kept intact so we can verify that the new notebook reproduces the known result.

For the deployment model, we now use **all 172,458 labeled pairs**. The threshold remains the value selected on the fixed validation split (`0.85`).

The TF-IDF vectorizers are refit on the complete labeled training text before fitting the production CatBoost model. This uses the labeled data available for final model training while preserving the exact feature recipe.

This means:

- validation model = exact `.989` reproduction
- production model = same architecture/features, retrained on all labeled pairs
- threshold = fixed from validation
- no test data is used in training


In [29]:
# ============================================================
# 13. PRODUCTION TF-IDF VECTORIZERS
# ============================================================

print("Fitting production TF-IDF vectorizers on all labeled pairs...")

all_s1_names = (
    training_enriched["s1_name"]
    .fill_null("")
    .to_list()
)

all_match_names = (
    training_enriched["match_name"]
    .fill_null("")
    .to_list()
)

all_s1_addresses = (
    training_enriched["s1_address"]
    .fill_null("")
    .to_list()
)

all_match_addresses = (
    training_enriched["match_address"]
    .fill_null("")
    .to_list()
)

name_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)

address_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)

char_name_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000,
)

char_address_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=150_000,
)

name_vectorizer.fit(all_s1_names + all_match_names)
address_vectorizer.fit(all_s1_addresses + all_match_addresses)
char_name_vectorizer.fit(all_s1_names + all_match_names)
char_address_vectorizer.fit(all_s1_addresses + all_match_addresses)

print("Name word vocabulary :", len(name_vectorizer.vocabulary_))
print("Address word vocabulary :", len(address_vectorizer.vocabulary_))
print("Name char vocabulary :", len(char_name_vectorizer.vocabulary_))
print("Address char vocabulary :", len(char_address_vectorizer.vocabulary_))
print("✓ Production vectorizers fitted.")


Fitting production TF-IDF vectorizers on all labeled pairs...
Name word vocabulary : 48854
Address word vocabulary : 100000
Name char vocabulary : 150000
Address char vocabulary : 150000
✓ Production vectorizers fitted.


In [30]:
# ============================================================
# 14. PRODUCTION V4 MATRIX
# ============================================================

print("Transforming all labeled text...")

all_name_s1_tfidf = name_vectorizer.transform(all_s1_names)
all_name_match_tfidf = name_vectorizer.transform(all_match_names)

all_address_s1_tfidf = address_vectorizer.transform(all_s1_addresses)
all_address_match_tfidf = address_vectorizer.transform(all_match_addresses)

all_char_name_s1 = char_name_vectorizer.transform(all_s1_names)
all_char_name_match = char_name_vectorizer.transform(all_match_names)

all_char_address_s1 = char_address_vectorizer.transform(all_s1_addresses)
all_char_address_match = char_address_vectorizer.transform(all_match_addresses)

all_name_tfidf_cosine = rowwise_cosine_similarity(
    all_name_s1_tfidf,
    all_name_match_tfidf,
)

all_address_tfidf_cosine = rowwise_cosine_similarity(
    all_address_s1_tfidf,
    all_address_match_tfidf,
)

all_name_char_tfidf_cosine = rowwise_cosine_similarity(
    all_char_name_s1,
    all_char_name_match,
)

all_address_char_tfidf_cosine = rowwise_cosine_similarity(
    all_char_address_s1,
    all_char_address_match,
)

X_v4_full = np.column_stack([
    X_v2.to_numpy(),
    all_name_tfidf_cosine,
    all_address_tfidf_cosine,
    all_name_char_tfidf_cosine,
    all_address_char_tfidf_cosine,
])

if X_v4_full.shape != (training_enriched.height, 18):
    raise RuntimeError(
        f"Unexpected production matrix shape: {X_v4_full.shape}"
    )

if not np.isfinite(X_v4_full).all():
    raise RuntimeError("Non-finite values found in production matrix.")

print("X_v4_full:", X_v4_full.shape)
print("✓ Production feature matrix passed.")


Transforming all labeled text...
X_v4_full: (172458, 18)
✓ Production feature matrix passed.


In [31]:
# ============================================================
# 15. TRAIN FINAL PRODUCTION CATBOOST
# ============================================================

FINAL_THRESHOLD = 0.85

final_catboost = CatBoostClassifier(
    iterations=1000,
    depth=10,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="Logloss",
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
    thread_count=-1,
)

print("Training production CatBoost on ALL labeled pairs...")
print("Rows:", X_v4_full.shape[0])
print("Features:", X_v4_full.shape[1])

train_start = time.time()

final_catboost.fit(
    X_v4_full,
    y_v2,
)

elapsed = time.time() - train_start

print(f"✓ Production model trained in {elapsed:.2f} seconds.")


Training production CatBoost on ALL labeled pairs...
Rows: 172458
Features: 18
✓ Production model trained in 65.95 seconds.


In [32]:
# ============================================================
# 16. DEPLOYMENT BUNDLE
# ============================================================

DEPLOYMENT_CONFIG = {
    "model_family": "CatBoostClassifier",
    "model_name": "V4 CatBoost Depth 10",
    "threshold": FINAL_THRESHOLD,
    "feature_count": 18,
    "features": V4_FEATURES,

    "catboost": {
        "iterations": 1000,
        "depth": 10,
        "learning_rate": 0.03,
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "l2_leaf_reg": 3.0,
        "random_seed": 42,
    },

    "word_tfidf": {
        "analyzer": "word",
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 100_000,
        "lowercase": True,
    },

    "char_tfidf": {
        "analyzer": "char",
        "ngram_range": [2, 5],
        "min_df": 2,
        "max_features": 150_000,
    },

    "validation_reproduction": {
        "threshold": VALIDATION_THRESHOLD,
        "precision": float(val_precision),
        "recall": float(val_recall),
        "f05": float(val_f05),
    },

    "training_rows": int(training_enriched.height),
}

deployment_bundle = {
    "model": final_catboost,

    "name_vectorizer": name_vectorizer,
    "address_vectorizer": address_vectorizer,
    "char_name_vectorizer": char_name_vectorizer,
    "char_address_vectorizer": char_address_vectorizer,

    "features": V4_FEATURES,
    "threshold": FINAL_THRESHOLD,
    "config": DEPLOYMENT_CONFIG,
}

print(json.dumps(DEPLOYMENT_CONFIG, indent=2))


{
  "model_family": "CatBoostClassifier",
  "model_name": "V4 CatBoost Depth 10",
  "threshold": 0.85,
  "feature_count": 18,
  "features": [
    "name_edit_similarity",
    "name_token_jaccard",
    "address_edit_similarity",
    "address_token_jaccard",
    "country_match",
    "name_exact_normalized",
    "name_exact_legal",
    "name_length_ratio",
    "address_exact_normalized",
    "address_length_ratio",
    "s1_name_missing",
    "match_name_missing",
    "s1_address_missing",
    "match_address_missing",
    "name_word_tfidf_cosine",
    "address_word_tfidf_cosine",
    "name_char_tfidf_cosine",
    "address_char_tfidf_cosine"
  ],
  "catboost": {
    "iterations": 1000,
    "depth": 10,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "l2_leaf_reg": 3.0,
    "random_seed": 42
  },
  "word_tfidf": {
    "analyzer": "word",
    "ngram_range": [
      1,
      2
    ],
    "min_df": 2,
    "max_features": 100000,
    "lowercase": true


In [33]:
# ============================================================
# 17. SAVE DEPLOYMENT ARTIFACTS
# ============================================================

MODEL_OUT = Path(
    "/kaggle/working/final_tuned_catboost_v4_depth10_production.joblib"
)

BUNDLE_OUT = Path(
    "/kaggle/working/final_tuned_catboost_v4_depth10_deployment_bundle.joblib"
)

CONFIG_OUT = Path(
    "/kaggle/working/final_tuned_catboost_v4_depth10_config.json"
)

joblib.dump(
    final_catboost,
    MODEL_OUT,
    compress=3,
)

joblib.dump(
    deployment_bundle,
    BUNDLE_OUT,
    compress=3,
)

CONFIG_OUT.write_text(
    json.dumps(DEPLOYMENT_CONFIG, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(" ", MODEL_OUT)
print(" ", BUNDLE_OUT)
print(" ", CONFIG_OUT)

print()
print("Sizes:")
print(f"  model  : {MODEL_OUT.stat().st_size / 1024**2:.2f} MB")
print(f"  bundle : {BUNDLE_OUT.stat().st_size / 1024**2:.2f} MB")
print(f"  config : {CONFIG_OUT.stat().st_size / 1024:.2f} KB")


Saved:
  /kaggle/working/final_tuned_catboost_v4_depth10_production.joblib
  /kaggle/working/final_tuned_catboost_v4_depth10_deployment_bundle.joblib
  /kaggle/working/final_tuned_catboost_v4_depth10_config.json

Sizes:
  model  : 2.38 MB
  bundle : 6.91 MB
  config : 1.29 KB


In [34]:
# ============================================================
# 18. FINAL ARTIFACT INTEGRITY CHECK
# ============================================================

loaded_bundle = joblib.load(BUNDLE_OUT)

required_bundle_keys = [
    "model",
    "name_vectorizer",
    "address_vectorizer",
    "char_name_vectorizer",
    "char_address_vectorizer",
    "features",
    "threshold",
    "config",
]

missing = [
    k for k in required_bundle_keys
    if k not in loaded_bundle
]

if missing:
    raise RuntimeError(f"Deployment bundle missing keys: {missing}")

if loaded_bundle["features"] != V4_FEATURES:
    raise RuntimeError("Feature ordering mismatch in deployment bundle.")

if loaded_bundle["threshold"] != FINAL_THRESHOLD:
    raise RuntimeError("Threshold mismatch in deployment bundle.")

if loaded_bundle["model"].get_feature_importance().shape[0] != 18:
    raise RuntimeError("Saved CatBoost model does not have 18 features.")

# A tiny in-memory inference sanity check.
probe = X_v4_full[:5]
probe_prob = loaded_bundle["model"].predict_proba(probe)[:, 1]

print("Probe probabilities:", probe_prob)
print()
print("✓ Deployment bundle reload succeeded.")
print("✓ Model has 18 features.")
print("✓ Feature ordering preserved.")
print("✓ Threshold preserved:", loaded_bundle["threshold"])
print("✓ All four TF-IDF vectorizers present.")
print()
print("=" * 65)
print("KAGGLE MODEL PACKAGING COMPLETE")
print("=" * 65)
print("Download this file for LOCAL inference:")
print(BUNDLE_OUT)


Probe probabilities: [0.99999629 0.98950953 0.99767906 0.99996496 0.99816657]

✓ Deployment bundle reload succeeded.
✓ Model has 18 features.
✓ Feature ordering preserved.
✓ Threshold preserved: 0.85
✓ All four TF-IDF vectorizers present.

KAGGLE MODEL PACKAGING COMPLETE
Download this file for LOCAL inference:
/kaggle/working/final_tuned_catboost_v4_depth10_deployment_bundle.joblib


## Local handoff

The local inference notebook should load **only**:

```python
deployment_bundle = joblib.load(
    "final_tuned_catboost_v4_depth10_deployment_bundle.joblib"
)
```

and use:

```python
deployment_bundle["model"]
deployment_bundle["name_vectorizer"]
deployment_bundle["address_vectorizer"]
deployment_bundle["char_name_vectorizer"]
deployment_bundle["char_address_vectorizer"]
deployment_bundle["features"]
deployment_bundle["threshold"]
```

The local inference code must recreate the same 18-feature order:

```text
1  name_edit_similarity
2  name_token_jaccard
3  address_edit_similarity
4  address_token_jaccard
5  country_match
6  name_exact_normalized
7  name_exact_legal
8  name_length_ratio
9  address_exact_normalized
10 address_length_ratio
11 s1_name_missing
12 match_name_missing
13 s1_address_missing
14 match_address_missing
15 name_tfidf_cosine
16 address_tfidf_cosine
17 name_char_tfidf_cosine
18 address_char_tfidf_cosine
```

**Do not regenerate the 64 candidate partitions.**

The next stage is local, partitioned inference over the existing candidate artifacts.
